In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

In [3]:
books_features = pd.read_csv("../Data/books_features.csv")

books_features.head()

,book_id,title,authors,original_publication_year,average_rating,ratings_count,image_url,tag_name
0,1,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,2008.0,4.34,4780653,https://images.gr-assets.com/books/1447303603m...,young-adult fiction dystopian dystopia fantasy...
1,2,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",1997.0,4.44,4602479,https://images.gr-assets.com/books/1474154022m...,fantasy young-adult fiction harry-potter ya se...
2,3,"Twilight (Twilight, #1)",Stephenie Meyer,2005.0,3.57,3866839,https://images.gr-assets.com/books/1361039443m...,young-adult fantasy vampires ya fiction parano...
3,4,To Kill a Mockingbird,Harper Lee,1960.0,4.25,3198671,https://images.gr-assets.com/books/1361975680m...,classics classic historical-fiction school clà...
4,5,The Great Gatsby,F. Scott Fitzgerald,1925.0,3.89,2683664,https://images.gr-assets.com/books/1490528560m...,classics fiction classic literature school his...


In [4]:
books_features.shape

(10000, 8)

In [6]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(books_features["tag_name"])

In [7]:
tfidf_matrix.shape

(10000, 2877)

In [8]:
content_similarity = cosine_similarity(tfidf_matrix)

In [9]:
content_similarity.shape

(10000, 10000)

In [10]:
title_to_index = pd.Series(
    books_features.index,
    index=books_features["title"]
).to_dict()

In [11]:
title_to_index["The Hunger Games (The Hunger Games, #1)"]

0

In [12]:
def recommend_content(title, n=10):
    
    if title not in title_to_index:
        return pd.DataFrame()
    
    idx = title_to_index[title]
    
    scores = list(enumerate(content_similarity[idx]))
    
    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )
    
    scores = [
        (book_idx, score)
        for book_idx, score in scores
        if book_idx != idx
    ][:n]
    
    recommendations = books_features.iloc[
        [x[0] for x in scores]
    ].copy()
    
    recommendations["similarity"] = [
        x[1] for x in scores
    ]
    
    return recommendations[
        ["book_id", "title", "authors", "similarity"]
    ]

In [13]:
recommend_content(
    "The Hunger Games (The Hunger Games, #1)"
)

,book_id,title,authors,similarity
16,17,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,0.946378
19,20,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,0.935665
506,507,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins,0.896548
11,12,"Divergent (Divergent, #1)",Veronica Roth,0.798836
2300,2301,"The Ask and the Answer (Chaos Walking, #2)",Patrick Ness,0.761434
780,781,"Specials (Uglies, #3)",Scott Westerfeld,0.760736
2562,2563,"Monsters of Men (Chaos Walking, #3)",Patrick Ness,0.757503
279,280,"Delirium (Delirium, #1)",Lauren Oliver,0.748676
518,519,"Pretties (Uglies, #2)",Scott Westerfeld,0.748150
1397,1398,"Extras (Uglies, #4)",Scott Westerfeld,0.741606
